# 03 - Enrich (language, experience, skills, role, category)

Task `Step_02_Enrich` of the pipeline job. Every stage works on pending rows only and is skipped when nothing is pending.

| Stage | Runs on | Method |
|---|---|---|
| 1. Rules | New / changed jobs (`job_key` + `description_hash` not yet enriched with this `enrichment_version`) | `langdetect`, regex experience + level, `ops.ref_skills` taxonomy (+ the parent skills in `implies`), title keyword rules (category hint, hard exclusion) |
| 2. Roles | Jobs without a role (or with an ambiguous one) that passed stage 1 | A role-matching text (title, rule skills and a 1500-character description slice starting at the first *Responsibilities / Requirements / Qualifications / What you will do / Must have / Key skills* heading, else the end of the description) and every role are embedded with `embedding_endpoint` (gte-large) via `ai_query`, cached in `ops.embedding_cache`. Cosine similarity + small bonuses (role alias in title, same category as the title rule). Close calls between roles of a different category or scope go to `role_llm_endpoint` (70B), which picks one of the top 5 candidates by number. **Category = the role's category**. Also rewrites `ops.role_similarity` (role-to-role cosine, used by the API for related roles) when the role texts change |
| Scope | Every row, one pass | Final `gold_eligible` / `gold_exclusion_reason` from the stage 1 reasons and the role's scope |
| 3. Skills LLM | Eligible jobs with an in-scope role and fewer than `min_skills` skills on a real description | `ai_query` structured JSON, grounded in the job text, cached in `ops.enrichment_llm_log` |

All model calls are capped per run (`embedding_max_rows`, `role_llm_max_rows`, `llm_max_rows`, 20 custom role checks) and cached, so a backlog is worked off over several runs and nothing is ever paid for twice. Jobs posted (else first seen) more than `job_max_age_days` (2, = `job_delete_after_days` in `04_gold_cleanup`) ago get no model calls, since gold drops them anyway. A failing LLM call is retried in later runs, at most 3 times per row. Within the caps, LinkedIn jobs and jobs in the categories of roles picked by app users (`ops.user_profile`) go first.

**Custom entries:** roles and skills users typed that are not in the reference lists. Skills are matched in job text as typed (group `Custom`; jobs enriched earlier are re-scanned once). Each new role is checked by `role_llm_endpoint` and stored in `ops.custom_roles`: `active` (new supported role, matched like the reference roles), `mapped` (same job as an existing role), `rejected` (outside the scope) or `pending` (retried next run, at most 3 times).

**Scope:** only roles with `ops.ref_roles.in_scope` (plus active custom roles) are supported. Stage 1 excludes a job by its title only when the title has a keyword of a category without in-scope roles and no keyword of an in-scope category nor a technical skill (e.g. "Senior Accountant"; "Marketing Data Analyst" is left to the embeddings), so no embedding / LLM calls are spent on it. A job with a role is in or out of scope by that role, re-checked on every run. Excluded jobs get `gold_exclusion_reason` `out_of_scope` and never reach gold. Title keyword changes apply to jobs enriched earlier only after an `enrichment_version` bump.

**Versions:** bumping `enrichment_version` re-runs stage 1 on every row. Roles of jobs whose description did not change are kept (gold keeps showing them) and re-scored in stage 2 as the embedding cap allows; cached embeddings and LLM answers are reused wherever their input is unchanged.

`role_method`: `embedding` / `embedding_ambiguous` (best embedding role kept, LLM tie-break pending) / `llm`.
`llm_status` (skills): `not_needed` / `pending` / `done` / `failed` / `skipped`.
No longer written (columns kept): `language_confidence`, `experience_text`, `category_method`, `role_alternative_score`.

**Job task environment dependencies:** `langdetect==1.0.9` (`model2vec` is no longer used)

In [ ]:
%run ./_common

In [ ]:
import hashlib
import json
import re

import numpy as np
from langdetect import DetectorFactory, LangDetectException, detect_langs
from pyspark.sql import Window
from pyspark.sql import functions as F

DetectorFactory.seed = 0

dbutils.widgets.text("enrichment_version", "v5", "Enrichment version (bump to re-run rules on everything)")
dbutils.widgets.dropdown("force_reenrich", "false", ["false", "true"], "Force re-enrich all silver rows")
dbutils.widgets.dropdown("force_rematch_roles", "false", ["false", "true"], "Re-score roles for all eligible jobs (uses caches)")
dbutils.widgets.text("min_skills", "3", "Fewer skills than this -> skills LLM")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en", "Embedding endpoint")
dbutils.widgets.text("embedding_max_rows", "400", "Max new job embeddings per run")
dbutils.widgets.text("role_min_margin", "0.02", "Best - 2nd role score below this -> LLM tie-break")
dbutils.widgets.text("role_llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "Role tie-break LLM endpoint")
dbutils.widgets.text("role_llm_max_rows", "50", "Max new role tie-break LLM calls per run")
dbutils.widgets.dropdown("llm_enabled", "true", ["true", "false"], "Use LLMs (skills + role tie-break)")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct", "Skills LLM endpoint")
dbutils.widgets.text("llm_max_rows", "150", "Max new skills LLM calls per run")
dbutils.widgets.text("job_max_age_days", "2", "Model calls only for jobs posted (else first seen) within N days")

ENRICHMENT_VERSION = dbutils.widgets.get("enrichment_version").strip()
FORCE_REENRICH = dbutils.widgets.get("force_reenrich") == "true"
FORCE_REMATCH_ROLES = dbutils.widgets.get("force_rematch_roles") == "true"
MIN_SKILLS = int(dbutils.widgets.get("min_skills"))
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint").strip()
EMBEDDING_MAX_ROWS = int(dbutils.widgets.get("embedding_max_rows"))
ROLE_MIN_MARGIN = float(dbutils.widgets.get("role_min_margin"))
ROLE_LLM_ENDPOINT = dbutils.widgets.get("role_llm_endpoint").strip()
ROLE_LLM_MAX_ROWS = int(dbutils.widgets.get("role_llm_max_rows"))
LLM_ENABLED = dbutils.widgets.get("llm_enabled") == "true"
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
LLM_MAX_ROWS = int(dbutils.widgets.get("llm_max_rows"))
JOB_MAX_AGE_DAYS = int(dbutils.widgets.get("job_max_age_days"))

LLM_MAX_CHARS = 4000          # description characters sent to the skills LLM
ROLE_SLICE_CHARS = 1500       # description characters in the role-matching text (embedded / sent for role tie-break)
ROLE_TEXT_SKILLS = 15         # rule skills listed in the role-matching text
TITLE_ALIAS_BONUS = 0.03      # role alias found in the job title
CATEGORY_RULE_BONUS = 0.03    # role category equals the title keyword category
ROLE_CANDIDATES = 5
MAX_ATTEMPTS = 3              # LLM calls per skills row, role tie-break or pending custom role before giving up
CUSTOM_ROLE_MAX_CHECKS = 20   # custom role LLM checks per run
PROMPT_VERSION = "p2"         # skills prompt; bump when LLM_PROMPT or RESPONSE_FORMAT change
ROLE_PROMPT_VERSION = "r2"    # role tie-break prompt; bump when ROLE_PROMPT changes (r2: numbered candidates)
SCOPE_REASON = "out_of_scope"
# Same cutoff as the gold source in 04_gold_cleanup (job_delete_after_days): older jobs never reach gold, so they get no
# embedding or LLM calls (silver alias s)
RECENT_JOB = f"coalesce(s.posted_date, to_date(s.first_seen_at)) >= date_sub(current_date(), {JOB_MAX_AGE_DAYS})"

for name, value in [("embedding_endpoint", EMBEDDING_ENDPOINT), ("role_llm_endpoint", ROLE_LLM_ENDPOINT),
                    ("llm_endpoint", LLM_ENDPOINT), ("enrichment_version", ENRICHMENT_VERSION)]:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", value):
        raise ValueError(f"Invalid {name}: {value!r}")

# All tables are created by 01_setup (checked by _common)
SILVER_TABLE = table("silver", "jobs_clean")
ENRICHED_TABLE = table("silver", "jobs_enriched")
LLM_LOG_TABLE = table("ops", "enrichment_llm_log")
EMBEDDING_CACHE_TABLE = table("ops", "embedding_cache")
ROLES_TABLE = table("ops", "ref_roles")
SKILLS_TABLE = table("ops", "ref_skills")
CUSTOM_ROLES_TABLE = table("ops", "custom_roles")
CUSTOM_SKILLS_TABLE = table("ops", "custom_skills")
PROFILE_TABLE = table("ops", "user_profile")
ROLE_SIMILARITY_TABLE = table("ops", "role_similarity")


def ai_result_fields(df, column):
    """Field names of an ai_query failOnError=>false struct, resolved from the schema instead of hard-coded."""
    fields = df.schema[column].dataType.fieldNames()
    result_field = "result" if "result" in fields else fields[0]
    error_field = next((name for name in fields if "error" in name.lower()), None)
    return result_field, error_field


print(f"version={ENRICHMENT_VERSION} embeddings={EMBEDDING_ENDPOINT} max={EMBEDDING_MAX_ROWS} "
      f"role_llm={ROLE_LLM_ENDPOINT}/{ROLE_PROMPT_VERSION} max={ROLE_LLM_MAX_ROWS} "
      f"skills_llm={LLM_ENDPOINT}/{PROMPT_VERSION} max={LLM_MAX_ROWS} llm_enabled={LLM_ENABLED}")

## Custom roles and skills from user profiles

In [ ]:
NEW_CUSTOM_SKILLS = []  # added in this run -> jobs enriched earlier are re-scanned after stage 1

with task_run("enrich_custom_entries") as metrics:
    ref_roles_df = spark.table(ROLES_TABLE)
    ref_role_rows = ref_roles_df.collect()
    scope_roles = sorted((row for row in ref_role_rows if row["in_scope"]), key=lambda row: row["role_title"])
    scope_titles = [row["role_title"] for row in scope_roles]
    scope_categories = sorted({row["category"] for row in scope_roles})

    # Everything already known (reference names, skill aliases and groups, checked custom entries) is skipped;
    # so are pending custom roles whose checks failed MAX_ATTEMPTS times
    custom_role_rows = spark.table(CUSTOM_ROLES_TABLE).select("role_title", "status", "attempts").collect()
    known_roles = {row["role_title"].lower() for row in ref_role_rows}
    known_roles |= {
        row["role_title"].lower() for row in custom_role_rows
        if row["status"] != "pending" or (row["attempts"] or 0) >= MAX_ATTEMPTS
    }
    pending_attempts = {row["role_title"].lower(): row["attempts"] or 0 for row in custom_role_rows if row["status"] == "pending"}
    known_skills = set()
    for row in spark.table(SKILLS_TABLE).collect():
        known_skills.add(row["skill"].lower())
        known_skills.update(alias.lower() for alias in row["aliases"] or [])
        if row["skill_group"]:
            known_skills.add(row["skill_group"].lower())
    known_skills |= {row["skill"].lower() for row in spark.table(CUSTOM_SKILLS_TABLE).select("skill").collect()}

    new_roles, new_skills = {}, {}  # lower(text) -> (text as typed, first user)
    for row in spark.table(PROFILE_TABLE).where("profile_id <> 'default'").select("profile_id", "target_roles", "skills").collect():
        for items, known, found in ((row["target_roles"], known_roles, new_roles), (row["skills"], known_skills, new_skills)):
            for item in items or []:
                text = (item or "").strip()
                if text and text.lower() not in known:
                    found.setdefault(text.lower(), (text, row["profile_id"]))

    if new_skills:
        (
            spark.createDataFrame(list(new_skills.values()), "skill STRING, first_user STRING")
            .withColumn("created_at", F.current_timestamp())
            .write.mode("append").saveAsTable(CUSTOM_SKILLS_TABLE)
        )
        NEW_CUSTOM_SKILLS = sorted(text for text, _ in new_skills.values())

    # New roles: one LLM check each (in scope? same as an existing role? category + description), least-tried first,
    # at most CUSTOM_ROLE_MAX_CHECKS per run (the rest are found again next run)
    to_check = sorted(new_roles, key=lambda key: (pending_attempts.get(key, 0), key))[:CUSTOM_ROLE_MAX_CHECKS]
    answers = {}
    if to_check and LLM_ENABLED:
        by_category = {}
        for row in ref_role_rows:
            by_category.setdefault(row["category"], []).append(row)
        area_lines = "\n".join(
            f"- {category}" + (
                "" if all(row["in_scope"] for row in by_category[category])
                else " (only: " + ", ".join(row["role_title"] for row in by_category[category] if row["in_scope"]) + ")"
            )
            for category in scope_categories
        )
        custom_role_prompt = (
            "A job board only supports job roles in these areas:\n" + area_lines + "\n\n"
            "A user typed a job role they are looking for. Answer in JSON:\n"
            "- in_scope: true only if it is a real job role in one of these areas (not a skill, tool, company, seniority or other field).\n"
            "- duplicate_of: the existing role below that is the same job (ignore seniority and wording), or none if it is a different job.\n"
            "- category: the area it belongs to.\n"
            "- description: one sentence about the work and typical tools, in the style of the existing roles.\n\n"
            "Existing roles:\n"
            + "\n".join(f"- {row['role_title']} ({row['category']}): {row['description']}" for row in scope_roles)
            + "\n\nUser role: "
        )
        custom_role_format = json.dumps({
            "type": "json_schema",
            "json_schema": {
                "name": "custom_role",
                "schema": {
                    "type": "object",
                    "properties": {
                        "in_scope": {"type": "boolean"},
                        "duplicate_of": {"type": "string", "enum": scope_titles + ["none"]},
                        "category": {"type": "string", "enum": scope_categories},
                        "description": {"type": "string"},
                    },
                    "required": ["in_scope", "duplicate_of", "category", "description"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        })
        assert "'" not in custom_role_format, "custom role response format must not contain single quotes"
        responses = spark.createDataFrame(
            [(key, custom_role_prompt + new_roles[key][0]) for key in to_check], "role_key STRING, request STRING"
        ).select(
            "role_key",
            F.expr(f"ai_query('{ROLE_LLM_ENDPOINT}', request, responseFormat => '{custom_role_format}', failOnError => false)").alias("r"),
        )
        result_field, _ = ai_result_fields(responses, "r")
        answers = {
            row["role_key"]: row["response"]
            for row in responses.select("role_key", F.col(f"r.{result_field}").cast("string").alias("response")).collect()
        }

    role_results = []
    for key in to_check:
        text, user = new_roles[key]
        try:
            answer = json.loads(answers.get(key) or "null") or {}
        except ValueError:
            answer = {}
        category = answer.get("category")
        description = (answer.get("description") or "").strip()[:300]
        duplicate = answer.get("duplicate_of")
        if not answer:
            status = "pending"
        elif not answer.get("in_scope"):
            status = "rejected"
        elif duplicate in scope_titles:
            status = "mapped"
        elif category in scope_categories and description:
            status = "active"
        else:
            status = "pending"
        role_results.append((
            text, status, category if status == "active" else None, description or None,
            duplicate if status == "mapped" else None, user, ROLE_LLM_ENDPOINT if answer else None, 1 if LLM_ENABLED else 0,
        ))

    if role_results:
        spark.createDataFrame(
            role_results,
            "role_title STRING, status STRING, category STRING, description STRING, duplicate_of STRING, first_user STRING, "
            "checked_by STRING, attempted INT",
        ).createOrReplaceTempView("custom_role_updates")
        spark.sql(f"""
            MERGE INTO {CUSTOM_ROLES_TABLE} t
            USING custom_role_updates s
            ON lower(t.role_title) = lower(s.role_title)
            WHEN MATCHED THEN UPDATE SET
              t.status = s.status, t.category = s.category, t.description = s.description, t.duplicate_of = s.duplicate_of,
              t.checked_by = s.checked_by, t.checked_at = current_timestamp(), t.attempts = coalesce(t.attempts, 0) + s.attempted
            WHEN NOT MATCHED THEN INSERT (role_title, status, category, description, duplicate_of, first_user, checked_by, created_at,
                                          checked_at, attempts)
              VALUES (s.role_title, s.status, s.category, s.description, s.duplicate_of, s.first_user, s.checked_by,
                      current_timestamp(), current_timestamp(), s.attempted)
        """)

    role_statuses = {status: sum(1 for result in role_results if result[1] == status) for status in {result[1] for result in role_results}}
    metrics.update(rows_in=len(new_roles) + len(new_skills), rows_out=len(role_results) + len(NEW_CUSTOM_SKILLS),
                   rows_rejected=role_statuses.get("rejected", 0))
    metrics["details"].update(new_roles=role_statuses, deferred_roles=len(new_roles) - len(to_check), new_skills=NEW_CUSTOM_SKILLS)
    print(f"custom roles checked: {role_statuses or 'none new'} (deferred {len(new_roles) - len(to_check)}) | "
          f"custom skills added: {NEW_CUSTOM_SKILLS or 'none'}")

## Reference data and rules

In [ ]:
categories = spark.table(table("ops", "ref_categories")).orderBy("priority").collect()
skills_ref = spark.table(SKILLS_TABLE).collect()
# Reference roles + custom roles the check above marked active
roles = (
    ref_roles_df.select("role_title", "category", "description", "aliases", "in_scope")
    .unionByName(
        spark.table(CUSTOM_ROLES_TABLE).where("status = 'active'")
        .select("role_title", "category", "description", F.array(F.lower("role_title")).alias("aliases"), F.lit(True).alias("in_scope"))
    )
    .orderBy("category", "role_title")
    .collect()
)
CATEGORY_NAMES = [row["category"] for row in categories]
ROLE_TITLES = [row["role_title"] for row in roles]
ROLE_CATEGORY = {row["role_title"]: row["category"] for row in roles}

# Product scope (01_setup: ops.ref_roles.in_scope)
IN_SCOPE_ROLES = sorted(row["role_title"] for row in roles if row["in_scope"])
IN_SCOPE_CATEGORIES = sorted({ROLE_CATEGORY[role] for role in IN_SCOPE_ROLES})
OUT_OF_SCOPE_CATEGORIES = sorted(set(CATEGORY_NAMES) - set(IN_SCOPE_CATEGORIES))
spark.createDataFrame([(role,) for role in IN_SCOPE_ROLES], "role_title STRING").createOrReplaceTempView("in_scope_roles")
print(f"scope: {len(IN_SCOPE_ROLES)} in-scope roles; categories without in-scope roles: {OUT_OF_SCOPE_CATEGORIES}")


def keyword_regex(keywords):
    alternatives = sorted({keyword.lower().strip() for keyword in keywords if keyword.strip()}, key=len, reverse=True)
    if not alternatives:
        return None
    return re.compile(r"(?<![a-z0-9])(?:" + "|".join(re.escape(a) for a in alternatives) + r")(?![a-z0-9])")


TITLE_RULES = [(row["category"], keyword_regex(row["title_keywords"])) for row in categories if row["title_keywords"]]
IN_SCOPE_TITLE_RE = keyword_regex([kw for row in categories if row["category"] in IN_SCOPE_CATEGORIES for kw in row["title_keywords"] or []])
OUT_OF_SCOPE_TITLE_RE = keyword_regex([kw for row in categories if row["category"] in OUT_OF_SCOPE_CATEGORIES for kw in row["title_keywords"] or []])
ROLE_ALIAS_RULES = [keyword_regex(row["aliases"]) if row["aliases"] else None for row in roles]
# Skills that appear in out-of-scope titles too ("SAP FICO Consultant", "Salesforce Marketing Cloud Specialist")
NON_SCOPE_SKILL_GROUPS = {"Business Tools"}


def clean_title(title):
    return re.sub(r"_+", " ", title or "")  # Workday titles like IN_Senior Associate_Data Engineering


def title_category(title):
    lowered = clean_title(title).lower()
    for category, pattern in TITLE_RULES:
        if pattern.search(lowered):
            return category
    return None


def title_out_of_scope(title, title_skills):
    """Hard exclusion by title: a keyword of a category without in-scope roles and nothing in scope (keyword or technical
    skill). Mixed titles ("Marketing Data Analyst", "Sales Engineer - Python") are left to the embeddings."""
    lowered = clean_title(title).lower()
    if OUT_OF_SCOPE_TITLE_RE is None or not OUT_OF_SCOPE_TITLE_RE.search(lowered):
        return False
    if IN_SCOPE_TITLE_RE is not None and IN_SCOPE_TITLE_RE.search(lowered):
        return False
    return not any(SKILL_GROUP.get(skill) not in NON_SCOPE_SKILL_GROUPS for skill in title_skills)


# Categories of the roles app users picked (ops.user_profile, written by the UI; the legacy 'default' row is ignored).
# Only the ORDER of the capped model-call queues changes: LinkedIn + these categories go first. Empty -> old order.
WANTED_ROLES = {
    row["role"] for row in spark.table(PROFILE_TABLE).where("profile_id <> 'default'")
    .select(F.explode("target_roles").alias("role")).distinct().collect()
    if row["role"]
}
WANTED_CATEGORIES = sorted({ROLE_CATEGORY[role] for role in WANTED_ROLES if role in ROLE_CATEGORY})
print(f"wanted roles={len(WANTED_ROLES)} categories={WANTED_CATEGORIES}")


def is_wanted(column):
    return F.col(column).isin(WANTED_CATEGORIES) if WANTED_CATEGORIES else F.lit(False)


# Skills: one regex over all aliases, longest first. A few common words need negative context.
ALIAS_TO_SKILL = {}
SKILL_GROUP = {}
SKILL_IMPLIES = {}  # skill -> parent skills (01_setup ref_skills.implies), e.g. Azure Data Factory -> Azure
for row in skills_ref:
    SKILL_GROUP[row["skill"]] = row["skill_group"]
    if row["implies"]:
        SKILL_IMPLIES[row["skill"]] = list(row["implies"])
    for alias in row["aliases"] or []:
        ALIAS_TO_SKILL.setdefault(alias.lower(), (row["skill"], row["skill_group"]))
for row in spark.table(CUSTOM_SKILLS_TABLE).select("skill").collect():  # skills users typed, matched as typed
    ALIAS_TO_SKILL.setdefault(row["skill"].lower(), (row["skill"], "Custom"))
    SKILL_GROUP.setdefault(row["skill"], "Custom")

NEGATIVE_CONTEXT = {
    "excel": r"(?!\s+(?:in|at|as)\b)",
    "spark": r"(?!\s+(?:greatness|joy|curiosity|creativity|innovation|conversations?|change|ideas?)\b)",
    "agentic": r"(?!\s+(?:marketing|commerce)\b)",  # WPP boilerplate: "agentic marketing platform"
}
SKILL_RE = re.compile(
    r"(?<![a-z0-9])("
    + "|".join(re.escape(alias) + NEGATIVE_CONTEXT.get(alias, "") for alias in sorted(ALIAS_TO_SKILL, key=len, reverse=True))
    + r")(?![a-z0-9])",
    re.IGNORECASE,
)


def extract_skills(text):
    skills, groups = [], []
    for match in SKILL_RE.finditer(text or ""):
        skill, group = ALIAS_TO_SKILL[match.group(1).lower()]
        if skill not in skills:
            skills.append(skill)
        if group not in groups:
            groups.append(group)
    # Implied parent skills go last, so a job asking for "Azure Data Factory" also matches a profile skill "Azure"
    for skill in list(skills):
        for parent in SKILL_IMPLIES.get(skill, []):
            if parent not in skills:
                skills.append(parent)
            if SKILL_GROUP.get(parent) and SKILL_GROUP[parent] not in groups:
                groups.append(SKILL_GROUP[parent])
    return skills, groups


# Experience
YEARS = r"(?:years?|yrs?)"
NUMBER_START = r"(?<![\d.])"  # whole numbers only: "1.5 years" is not 5, "200 years" is not 0
EXPERIENCE_PATTERNS = [
    ("range", re.compile(rf"{NUMBER_START}(\d{{1,2}})\s*\+?\s*(?:-|–|—|~|to)\s*(\d{{1,2}})\s*\+?\s*{YEARS}", re.I)),
    ("min", re.compile(rf"(?:minimum|min\.?|at least|over|more than)\s*(?:of\s*)?(\d{{1,2}})\s*\+?\s*{YEARS}", re.I)),
    ("plus", re.compile(rf"{NUMBER_START}(\d{{1,2}})\s*(?:\+|plus)\s*{YEARS}", re.I)),
    ("plain", re.compile(rf"{NUMBER_START}(\d{{1,2}})\s*{YEARS}", re.I)),
]
EXPERIENCE_WORD = re.compile(r"experience|\bexp\b|\bexp\.", re.I)
EXCLUDE_CONTEXT = re.compile(r"education|degree|years? old|\bage\b|anniversary|founded|history|warranty|since \d{4}", re.I)
# Company boilerplate ("a firm with over 25 years of experience serving clients") is not a requirement, unless the
# sentence is about the candidate ("candidates with over 5 years", "a Data Scientist with over 4-8 years")
BOILERPLATE_BEFORE = re.compile(r"\b(?:with|for)\s+over\b|\bsince\b|\bserving\b|\bestablished\b", re.I)
PERSON_BEFORE = re.compile(
    r"\b(?:candidates?|applicants?|professionals?|individuals?|someone|you|seeking|looking for|hiring"
    r"|engineers?|developers?|analysts?|scientists?|architects?|consultants?|administrators?|testers?)\b",
    re.I,
)
MAX_MIN_YEARS = 20  # higher minimums are boilerplate in practice
FRESHER_RE = re.compile(r"\bfreshers?\b", re.I)
FRESHER_NEGATIVE = re.compile(
    r"\b(?:no|not|non)[\s-]+(?:for\s+)?freshers?\b|\bfreshers?\s+(?:(?:are|is)\s+)?(?:not|need not|cannot|can not|should not|please do not)\b",
    re.I,
)
FRESHER_POSITIVE = re.compile(
    r"\bfreshers?\s+(?:(?:are|is)\s+)?(?:can|may|welcome|eligible|encouraged|apply|preferred|also)\b"
    r"|\b(?:open to|welcomes?|for|hiring|including|suitable for|calling)\s+(?:all\s+|the\s+)?freshers?\b"
    r"|\bfreshers?\s*(?:/|or|and|&)\s*experienced\b|\bexperienced\s*(?:/|or|and|&)\s*freshers?\b"
    r"|\bexperience\s*[:-]\s*freshers?\b|\bno (?:prior )?experience (?:is )?required\b",
    re.I,
)


def is_fresher(title, text):
    """'fresher' only counts in the title or with positive context ("freshers can apply"), never in "no freshers"."""
    if FRESHER_RE.search(clean_title(title)):
        return True
    return not FRESHER_NEGATIVE.search(text) and bool(FRESHER_POSITIVE.search(text))


def extract_experience(text, title=""):
    text = text or ""
    candidates = []
    for priority, (kind, pattern) in enumerate(EXPERIENCE_PATTERNS):
        for match in pattern.finditer(text):
            window = text[max(0, match.start() - 80): match.end() + 80]
            close = text[max(0, match.start() - 30): match.end() + 30]
            if not EXPERIENCE_WORD.search(window) or EXCLUDE_CONTEXT.search(close):
                continue
            prefix = re.split(r"[.!?;\n]", text[max(0, match.start() - 40): match.start()])[-1]  # same sentence only
            before = prefix + match.group(0)
            if BOILERPLATE_BEFORE.search(before) and not PERSON_BEFORE.search(before):
                continue
            low = int(match.group(1))
            high = int(match.group(2)) if kind == "range" else None
            if high is not None and high < low:
                low, high = high, low
            if low > MAX_MIN_YEARS or (high is not None and high > 30):
                continue
            candidates.append((match.start(), priority, low, high))
    if candidates:
        _, _, low, high = min(candidates)
        return low, high
    if is_fresher(title, text):
        return 0, 1
    return None, None


LEVEL_RULES = [
    ("Intern", re.compile(r"\b(?:intern|internship|trainee|apprentice)\b", re.I)),
    ("Director+", re.compile(r"\b(?:director|vp|vice president|head of|chief|cto|cio|cdo)\b", re.I)),
    ("Principal/Staff", re.compile(r"\b(?:principal|staff|distinguished|architect)\b", re.I)),
    ("Lead/Manager", re.compile(r"\b(?:lead|manager|supervisor)\b", re.I)),
    ("Senior", re.compile(r"\b(?:senior|sr\b\.?|iii|iv)\b", re.I)),
    ("Junior", re.compile(r"\b(?:junior|jr\b\.?|associate|graduate|fresher|entry)\b", re.I)),
]
LINKEDIN_LEVELS = {"internship": "Intern", "entry level": "Entry", "associate": "Junior", "mid-senior level": "Mid-Senior",
                   "director": "Director+", "executive": "Director+"}


def years_level(min_years):
    if min_years is None:
        return None
    if min_years < 1:
        return "Entry"
    if min_years < 3:
        return "Junior"
    if min_years < 6:
        return "Mid"
    if min_years < 10:
        return "Senior"
    return "Lead/Manager"


def experience_level(title, seniority_level, min_years):
    for level, pattern in LEVEL_RULES:
        if pattern.search(title or ""):
            return level
    return years_level(min_years) or LINKEDIN_LEVELS.get((seniority_level or "").lower())


def detect_language(text):
    sample = (text or "")[:1500]
    if len(sample) < 100:
        return None  # too short to be reliable -> treated as unknown (not excluded)
    try:
        return detect_langs(sample)[0].lang
    except LangDetectException:
        return None


# Role-matching text: title, rule skills and the part of the description about the work (company intros and benefits
# usually come before the first heading). Without a heading, or with one in the last 1500 characters: the end of the description.
ROLE_SECTION_RE = re.compile(r"responsibilities|requirements|qualifications|what you(?:'ll| will) do|must[- ]haves?|key skills", re.I)


def role_text(title, skills, description):
    description = description or ""
    tail_start = max(0, len(description) - ROLE_SLICE_CHARS)
    match = ROLE_SECTION_RE.search(description)
    start = min(match.start(), tail_start) if match else tail_start
    parts = [clean_title(title).strip()]
    if skills:
        parts.append("Skills: " + ", ".join(skills[:ROLE_TEXT_SKILLS]))
    parts.append(description[start: start + ROLE_SLICE_CHARS].strip())
    return ". ".join(part for part in parts if part)


print(f"{len(TITLE_RULES)} title rules, {len(ALIAS_TO_SKILL)} skill aliases ({len(SKILL_IMPLIES)} with parents), {len(ROLE_TITLES)} roles")

## Stage 1 - rules on new / changed jobs

In [ ]:
ENRICHED_SCHEMA = (
    "job_key STRING, description_hash STRING, language STRING, "
    "experience_min_years INT, experience_max_years INT, experience_level STRING, "
    "skills ARRAY<STRING>, skill_groups ARRAY<STRING>, category STRING, category_score DOUBLE, "
    "gold_eligible BOOLEAN, gold_exclusion_reason STRING, llm_status STRING, llm_attempts INT, "
    "role_title STRING, role_score DOUBLE, role_alternative STRING, role_method STRING, "
    "enrichment_version STRING, enriched_at TIMESTAMP, _job_run_id STRING"
)
ENRICHED_COLUMNS = [part.strip().split(" ")[0] for part in ENRICHED_SCHEMA.split(", ")]
# Kept when the description did not change and the job still passes stage 1 (e.g. an enrichment_version bump): gold keeps
# the role until stage 2 re-scores it (role_method is reset to NULL)
KEEP_ROLE_COLUMNS = {"role_title", "role_score", "role_alternative", "category", "category_score"}
KEEP_ROLE = "t.description_hash <=> s.description_hash AND s.gold_eligible AND t.role_title IS NOT NULL"


def enrich_rows(rows):
    """Rule-based enrichment. Role columns are NULL here; stage 2 fills them for eligible jobs."""
    tuples = []
    for row in rows:
        title = clean_title(row["title"])
        description = row["description"] or ""

        language = detect_language(description)
        min_years, max_years = extract_experience(description, title)
        skills, skill_groups = extract_skills(f"{title}. {description}")
        title_skills, _ = extract_skills(title)
        category = title_category(title)

        exclusion = []
        if not row["is_india"]:
            exclusion.append("not_india")
        if language not in (None, "en"):
            exclusion.append(f"non_english:{language}")
        if title_out_of_scope(title, title_skills):
            exclusion.append(SCOPE_REASON)  # e.g. "Senior Accountant": no embedding or LLM calls
        eligible = not exclusion

        few_skills = len(skills) < MIN_SKILLS and len(description) >= 300
        output = {
            "job_key": row["job_key"],
            "description_hash": row["description_hash"],
            "language": language,
            "experience_min_years": min_years,
            "experience_max_years": max_years,
            "experience_level": experience_level(title, row["seniority_level"], min_years),
            "skills": skills,
            "skill_groups": skill_groups,
            "category": category,  # title-rule hint; replaced by the role's category in stage 2
            "category_score": 1.0 if category else None,
            "gold_eligible": eligible,
            "gold_exclusion_reason": ",".join(exclusion) or None,
            "llm_status": "skipped" if not eligible else ("pending" if few_skills else "not_needed"),
            "llm_attempts": 0,
            "role_title": None, "role_score": None, "role_alternative": None, "role_method": None,
            "enrichment_version": ENRICHMENT_VERSION,
            "enriched_at": None,
            "_job_run_id": JOB_RUN_ID,
        }
        tuples.append(tuple(output[column] for column in ENRICHED_COLUMNS))
    return tuples

In [ ]:
with task_run("enrich") as metrics:
    silver = spark.table(SILVER_TABLE).select("job_key", "description_hash", "title", "description", "is_india", "seniority_level")
    if FORCE_REENRICH:
        pending = silver
    else:
        already = spark.table(ENRICHED_TABLE).where(F.col("enrichment_version") == ENRICHMENT_VERSION).select("job_key", "description_hash")
        pending = silver.join(already, ["job_key", "description_hash"], "left_anti")

    pending_rows = pending.collect()
    enriched_tuples = enrich_rows(pending_rows)
    metrics["rows_in"] = len(pending_rows)

    if enriched_tuples:
        (
            spark.createDataFrame(enriched_tuples, ENRICHED_SCHEMA)
            .withColumn("enriched_at", F.current_timestamp())
            .createOrReplaceTempView("enriched_updates")
        )
        assignments = ", ".join(
            f"t.{c} = CASE WHEN {KEEP_ROLE} THEN t.{c} ELSE s.{c} END" if c in KEEP_ROLE_COLUMNS else f"t.{c} = s.{c}"
            for c in ENRICHED_COLUMNS if c != "job_key"
        )
        spark.sql(f"""
            MERGE INTO {ENRICHED_TABLE} t
            USING enriched_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED THEN UPDATE SET {assignments}
            WHEN NOT MATCHED THEN INSERT ({", ".join(ENRICHED_COLUMNS)}) VALUES ({", ".join(f"s.{c}" for c in ENRICHED_COLUMNS)})
        """)

    status_index = ENRICHED_COLUMNS.index("llm_status")
    reason_index = ENRICHED_COLUMNS.index("gold_exclusion_reason")
    metrics["details"]["stage1"] = {
        "rows": len(enriched_tuples),
        "eligible": sum(1 for t in enriched_tuples if t[ENRICHED_COLUMNS.index("gold_eligible")]),
        "title_out_of_scope": sum(1 for t in enriched_tuples if SCOPE_REASON in (t[reason_index] or "")),
        "llm_status": {s: sum(1 for t in enriched_tuples if t[status_index] == s) for s in {t[status_index] for t in enriched_tuples}},
    }
    print(f"Stage 1 enriched {len(enriched_tuples)} rows: {metrics['details']['stage1']}")

In [ ]:
# Custom skills added in this run: add them to jobs enriched before they existed (stage 1 only sees new / changed jobs)
if NEW_CUSTOM_SKILLS:
    with task_run("enrich_custom_skills") as metrics:
        canonical = {skill.lower(): skill for skill in NEW_CUSTOM_SKILLS}
        new_skill_re = re.compile(
            r"(?<![a-z0-9])(" + "|".join(re.escape(skill) for skill in sorted(canonical, key=len, reverse=True)) + r")(?![a-z0-9])",
            re.IGNORECASE,
        )
        scan_rows = spark.sql(f"""
            SELECT e.job_key, e.description_hash, s.title, s.description
            FROM {ENRICHED_TABLE} e
            JOIN {SILVER_TABLE} s ON s.job_key = e.job_key AND s.description_hash = e.description_hash
            WHERE e.gold_eligible
        """).collect()
        skill_updates = []
        for row in scan_rows:
            found = sorted({canonical[match.group(1).lower()] for match in new_skill_re.finditer(f"{clean_title(row['title'])}. {row['description'] or ''}")})
            if found:
                skill_updates.append((row["job_key"], row["description_hash"], found))
        if skill_updates:
            spark.createDataFrame(skill_updates, "job_key STRING, description_hash STRING, found ARRAY<STRING>").createOrReplaceTempView("custom_skill_updates")
            spark.sql(f"""
                MERGE INTO {ENRICHED_TABLE} t
                USING custom_skill_updates s
                ON t.job_key = s.job_key AND t.description_hash = s.description_hash
                WHEN MATCHED THEN UPDATE SET
                  t.skills = array_distinct(concat(coalesce(t.skills, CAST(array() AS ARRAY<STRING>)), s.found)),
                  t.skill_groups = array_distinct(concat(coalesce(t.skill_groups, CAST(array() AS ARRAY<STRING>)), array('Custom')))
            """)
        metrics.update(rows_in=len(scan_rows), rows_out=len(skill_updates), rows_rejected=0)
        metrics["details"].update(new_custom_skills=NEW_CUSTOM_SKILLS)
        print(f"custom skills {NEW_CUSTOM_SKILLS}: added to {len(skill_updates)} of {len(scan_rows)} existing jobs")

## Stage 2 - role matching (large embeddings + LLM tie-break)

In [ ]:
def append_llm_log(responses, endpoint, prompt_version, response=None):
    """Appends ai_query responses (job_key, description_hash, struct r) to the LLM log; this runs the billable calls.
    response(raw) can derive the stored response from the raw result (default: the raw result)."""
    result_field, error_field = ai_result_fields(responses, "r")
    raw = F.col(f"r.{result_field}").cast("string")
    (
        responses.select(
            "job_key", "description_hash", F.lit(endpoint).alias("endpoint"), F.lit(prompt_version).alias("prompt_version"),
            (response(raw) if response else raw).alias("response"),
            (F.col(f"r.{error_field}").cast("string") if error_field else F.lit(None).cast("string")).alias("error"),
            F.current_timestamp().alias("called_at"), F.lit(JOB_RUN_ID).alias("_job_run_id"),
        )
        .write.mode("append").saveAsTable(LLM_LOG_TABLE)
    )


def embed_missing(texts_df, max_rows):
    """Embeds texts (cache_key, text, text_hash) not yet cached for this model. Returns (new calls, failures)."""
    cached = (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNotNull())
        .select("text_hash").distinct()
    )
    todo = texts_df.join(cached, "text_hash", "left_anti").dropDuplicates(["text_hash"])
    if max_rows is not None:
        if "priority" in texts_df.columns:  # 0 = wanted first
            todo = todo.orderBy("priority")
        todo = todo.limit(max_rows)
    # Materialise the batch first so the billable calls run on a fixed set of texts exactly once
    todo_rows = todo.select("cache_key", "text", "text_hash").collect()
    if not todo_rows:
        return 0, 0

    responses = spark.createDataFrame(todo_rows, "cache_key STRING, text STRING, text_hash STRING").select(
        "text_hash", "cache_key",
        F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', text, failOnError => false)").alias("r"),
    )
    result_field, error_field = ai_result_fields(responses, "r")
    (
        responses.select(
            "text_hash", F.lit(EMBEDDING_ENDPOINT).alias("model"), "cache_key",
            F.col(f"r.{result_field}").cast("array<float>").alias("embedding"),
            (F.col(f"r.{error_field}").cast("string") if error_field else F.lit(None).cast("string")).alias("error"),
            F.current_timestamp().alias("created_at"), F.lit(JOB_RUN_ID).alias("_job_run_id"),
        )
        .write.mode("append").saveAsTable(EMBEDDING_CACHE_TABLE)
    )
    failures = (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("_job_run_id") == JOB_RUN_ID) & (F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNull())
        .where(F.col("text_hash").isin([row["text_hash"] for row in todo_rows]))
        .count()
    )
    return len(todo_rows), failures


def cached_embeddings():
    latest = Window.partitionBy("text_hash").orderBy(F.col("created_at").desc())
    return (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNotNull())
        .withColumn("_rank", F.row_number().over(latest)).where("_rank = 1")
        .select("text_hash", "embedding")
    )


def normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1, norms)


def table_property(table_name, key):
    return {row["key"]: row["value"] for row in spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()}.get(key)


ROLE_PROMPT = (
    "Choose the standard role that best matches this job posting. Consider the responsibilities and required skills in the "
    "description, not only the title. Return JSON with choice set to the number of the best candidate role.\n"
)
ROLE_RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "role_choice",
        "schema": {
            "type": "object",
            "properties": {"choice": {"type": "string", "enum": [str(number) for number in range(1, ROLE_CANDIDATES + 1)]}},
            "required": ["choice"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})
assert "'" not in ROLE_RESPONSE_FORMAT, "role response format must not contain single quotes"


def role_answer(raw):
    """Stores the chosen candidate's title next to its number, so a cached answer stays valid if the candidate order changes."""
    choice = F.get_json_object(raw, "$.choice")
    title = F.try_element_at("candidates", F.when(choice.rlike("^[1-9]$"), choice.cast("int")))
    return F.when(title.isNotNull(), F.to_json(F.struct(choice.alias("choice"), title.alias("role_title")))).otherwise(raw)


ROLE_DESCRIPTIONS = {row["role_title"]: row["description"] for row in roles}
ROLE_IN_SCOPE = {row["role_title"]: row["in_scope"] for row in roles}
ROLE_ALIAS_BY_TITLE = dict(zip(ROLE_TITLES, ROLE_ALIAS_RULES))

In [ ]:
with task_run("enrich_roles") as metrics:
    # Role texts: embedded once (cached unless the text changes). ops.role_similarity covers the roles users can pick and
    # is rewritten only when their texts (or the embedding model) change.
    role_texts = {row["role_title"]: f"{row['role_title']}. {row['description']}" for row in roles}
    similarity_titles = sorted(title for title in ROLE_TITLES if ROLE_IN_SCOPE[title])
    role_hash = hashlib.sha256(
        json.dumps([EMBEDDING_ENDPOINT, [[title, role_texts[title]] for title in similarity_titles]], ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    similarity_current = table_property(ROLE_SIMILARITY_TABLE, "jobseeker.role_hash") == role_hash

    # Jobs that need a role: passed stage 1 (or excluded only by their role's scope) without a role / with an ambiguous one
    role_filter = "TRUE" if FORCE_REMATCH_ROLES else "(e.role_method IS NULL OR e.role_method = 'embedding_ambiguous')"
    need_rows = spark.sql(f"""
        SELECT e.job_key, e.description_hash, s.title, s.description, s.ats_type, e.category AS hint_category,
               e.role_title IS NOT NULL AS has_role
        FROM {ENRICHED_TABLE} e
        JOIN {SILVER_TABLE} s ON s.job_key = e.job_key AND s.description_hash = e.description_hash
        WHERE (e.gold_eligible OR (e.role_title IS NOT NULL AND e.gold_exclusion_reason = '{SCOPE_REASON}')) AND {role_filter}
          AND {RECENT_JOB}
    """).collect()

    if not need_rows and similarity_current:
        metrics.update(rows_in=0, rows_out=0, rows_rejected=0, message="nothing pending")
        print("roles: no job needs a role and ops.role_similarity is current - skipped")
    else:
        role_df = spark.createDataFrame(
            [(f"role:{title}", text) for title, text in role_texts.items()], "cache_key STRING, text STRING"
        ).withColumn("text_hash", F.sha2("text", 256))
        new_role_embeddings, role_embedding_failures = embed_missing(role_df, None)

        # Role-matching texts from the rules only (not LLM skills), so the text and its cached embedding stay stable
        job_inputs, priorities = {}, {}
        for row in need_rows:
            key = (row["job_key"], row["description_hash"])
            skills, _ = extract_skills(f"{clean_title(row['title'])}. {row['description'] or ''}")
            job_inputs[key] = role_text(row["title"], skills, row["description"])
            # 0 = no role yet, LinkedIn / wanted category ... 3 = role kept from an earlier version, other
            preferred = (row["ats_type"] or "").lower() == "linkedin" or row["hint_category"] in WANTED_CATEGORIES
            priorities[key] = (2 if row["has_role"] else 0) + (0 if preferred else 1)
        new_job_embeddings = job_embedding_failures = 0
        if need_rows:
            need = spark.createDataFrame(
                [(key[0], key[1], text, priorities[key]) for key, text in job_inputs.items()],
                "job_key STRING, description_hash STRING, text STRING, priority INT",
            ).withColumn("text_hash", F.sha2("text", 256))
            new_job_embeddings, job_embedding_failures = embed_missing(
                need.select(F.col("job_key").alias("cache_key"), "text", "text_hash", "priority"), EMBEDDING_MAX_ROWS
            )

        embeddings = cached_embeddings()
        role_vectors = {
            row["cache_key"][len("role:"):]: row["embedding"]
            for row in role_df.join(embeddings, "text_hash").select("cache_key", "embedding").collect()
        }
        usable_roles = [row for row in roles if row["role_title"] in role_vectors]
        if len(usable_roles) < 2:
            raise RuntimeError(f"Only {len(usable_roles)} role embeddings available - check the embedding endpoint {EMBEDDING_ENDPOINT}")
        role_matrix = normalize(np.array([role_vectors[row["role_title"]] for row in usable_roles], dtype=np.float32))
        role_alias_rules = [ROLE_ALIAS_BY_TITLE[row["role_title"]] for row in usable_roles]
        role_categories = [row["category"] for row in usable_roles]

        # ops.role_similarity: both directions, role_a = role_b included with 1.0. The hash is only stored when every role
        # had an embedding, so a partial table is rewritten on the next run.
        similarity_rows = 0
        if not similarity_current:
            indexes = [index for index, row in enumerate(usable_roles) if row["in_scope"]]
            sims = role_matrix[indexes] @ role_matrix[indexes].T
            pairs = [
                (usable_roles[a]["role_title"], usable_roles[b]["role_title"], 1.0 if a == b else round(float(sims[i, j]), 4))
                for i, a in enumerate(indexes) for j, b in enumerate(indexes)
            ]
            spark.createDataFrame(pairs, "role_a STRING, role_b STRING, sim DOUBLE").createOrReplaceTempView("role_similarity_updates")
            spark.sql(f"INSERT OVERWRITE {ROLE_SIMILARITY_TABLE} SELECT role_a, role_b, sim FROM role_similarity_updates")
            if len(indexes) == len(similarity_titles):
                spark.sql(f"ALTER TABLE {ROLE_SIMILARITY_TABLE} SET TBLPROPERTIES ('jobseeker.role_hash' = '{role_hash}')")
            similarity_rows = len(pairs)
            print(f"ops.role_similarity rewritten: {len(indexes)} of {len(similarity_titles)} roles, {similarity_rows} rows")

        job_rows = (
            need.join(embeddings, "text_hash").select("job_key", "description_hash", "embedding").collect() if need_rows else []
        )
        titles = {(row["job_key"], row["description_hash"]): row["title"] for row in need_rows}

        scored = {}
        if job_rows:
            job_matrix = normalize(np.array([row["embedding"] for row in job_rows], dtype=np.float32))
            similarity = job_matrix @ role_matrix.T
            for index, row in enumerate(job_rows):
                key = (row["job_key"], row["description_hash"])
                title = clean_title(titles[key]).lower()
                rule_category = title_category(titles[key])  # recomputed: e.category may already hold a previous role's category
                scores = similarity[index].astype(float).copy()
                for role_index, alias_rule in enumerate(role_alias_rules):
                    if alias_rule is not None and alias_rule.search(title):
                        scores[role_index] += TITLE_ALIAS_BONUS
                    if rule_category and role_categories[role_index] == rule_category:
                        scores[role_index] += CATEGORY_RULE_BONUS
                order = np.argsort(scores)[::-1]
                best, second = usable_roles[order[0]], usable_roles[order[1]]
                # A close call only matters when it can change the category or the scope; otherwise the best role is kept
                ambiguous = (
                    (scores[order[0]] - scores[order[1]]) < ROLE_MIN_MARGIN
                    and (best["category"] != second["category"] or best["in_scope"] != second["in_scope"])
                )
                candidates = [usable_roles[i]["role_title"] for i in order[:ROLE_CANDIDATES]]
                scored[key] = {
                    "role_title": best["role_title"], "role_score": round(float(scores[order[0]]), 4),
                    "role_alternative": second["role_title"],
                    "role_method": "embedding_ambiguous" if ambiguous else "embedding",
                    "candidates": candidates,
                    "candidate_scores": {usable_roles[i]["role_title"]: round(float(scores[i]), 4) for i in order[:ROLE_CANDIDATES]},
                }

        # LLM tie-break for ambiguous jobs: cached answers first, then new calls up to the cap; MAX_ATTEMPTS calls per job
        ambiguous_keys = [key for key, value in scored.items() if value["role_method"] == "embedding_ambiguous"]
        role_llm_calls = role_llm_failures = role_llm_cache_hits = role_llm_given_up = 0
        if LLM_ENABLED and ambiguous_keys:
            this_prompt = (F.col("endpoint") == ROLE_LLM_ENDPOINT) & (F.col("prompt_version") == ROLE_PROMPT_VERSION)
            ambiguous_df = spark.createDataFrame(ambiguous_keys, "job_key STRING, description_hash STRING")

            def logged_answers():
                """(job_key, description_hash) -> (role title of the newest usable answer, its run, calls logged so far)"""
                per_job = Window.partitionBy("job_key", "description_hash")
                return {
                    (row["job_key"], row["description_hash"]): (row["answer"], row["_job_run_id"], row["calls"])
                    for row in spark.table(LLM_LOG_TABLE).where(this_prompt)
                    .join(F.broadcast(ambiguous_df), ["job_key", "description_hash"])
                    .withColumn("answer", F.when(F.col("error").isNull(),
                                                 F.from_json("response", "struct<choice:string, role_title:string>").role_title))
                    .withColumn("calls", F.count(F.lit(1)).over(per_job))
                    .withColumn("_rank", F.row_number().over(per_job.orderBy(F.col("answer").isNotNull().desc(), F.col("called_at").desc())))
                    .where("_rank = 1")
                    .select("job_key", "description_hash", "answer", "_job_run_id", "calls")
                    .collect()
                }

            def answered(key):
                return answers.get(key, (None, None, 0))[0] in scored[key]["candidates"]

            answers = logged_answers()
            to_call = sorted(  # LinkedIn / wanted categories first (stable sort keeps the rest in their old order)
                (key for key in ambiguous_keys if not answered(key) and answers.get(key, (None, None, 0))[2] < MAX_ATTEMPTS),
                key=lambda key: (priorities[key], ROLE_CATEGORY.get(scored[key]["role_title"]) not in WANTED_CATEGORIES),
            )[:ROLE_LLM_MAX_ROWS]
            if to_call:
                requests = spark.createDataFrame(
                    [
                        (
                            key[0], key[1], scored[key]["candidates"],
                            ROLE_PROMPT
                            + "\nCandidate roles:\n"
                            + "\n".join(f"{number}. {title}: {ROLE_DESCRIPTIONS[title]}"
                                        for number, title in enumerate(scored[key]["candidates"], start=1))
                            + f"\n\nJob posting:\n{job_inputs[key]}",
                        )
                        for key in to_call
                    ],
                    "job_key STRING, description_hash STRING, candidates ARRAY<STRING>, request STRING",
                )
                responses = requests.select(
                    "job_key", "description_hash", "candidates",
                    F.expr(f"ai_query('{ROLE_LLM_ENDPOINT}', request, responseFormat => '{ROLE_RESPONSE_FORMAT}', failOnError => false)").alias("r"),
                )
                append_llm_log(responses, ROLE_LLM_ENDPOINT, ROLE_PROMPT_VERSION, response=role_answer)
                role_llm_calls = len(to_call)
                answers = logged_answers()
                role_llm_failures = sum(1 for key in to_call if not answered(key))

            for key in ambiguous_keys:
                answer, answer_run, calls = answers.get(key, (None, None, 0))
                value = scored[key]
                if answered(key):
                    if answer_run != JOB_RUN_ID:
                        role_llm_cache_hits += 1
                    if answer != value["role_title"]:
                        value["role_alternative"] = value["role_title"]
                        value["role_title"] = answer
                        value["role_score"] = value["candidate_scores"][answer]
                    value["role_method"] = "llm"
                elif calls >= MAX_ATTEMPTS:
                    value["role_method"] = "embedding"  # tie-break given up: keep the best embedding role
                    role_llm_given_up += 1

        # Write roles; the job category becomes the role's category
        if scored:
            (
                spark.createDataFrame(
                    [
                        (key[0], key[1], value["role_title"], value["role_score"], value["role_alternative"],
                         value["role_method"], ROLE_CATEGORY[value["role_title"]])
                        for key, value in scored.items()
                    ],
                    "job_key STRING, description_hash STRING, role_title STRING, role_score DOUBLE, role_alternative STRING, "
                    "role_method STRING, role_category STRING",
                )
                .createOrReplaceTempView("role_updates")
            )
            spark.sql(f"""
                MERGE INTO {ENRICHED_TABLE} t
                USING role_updates s
                ON t.job_key = s.job_key AND t.description_hash = s.description_hash
                WHEN MATCHED THEN UPDATE SET
                  t.role_title = s.role_title,
                  t.role_score = s.role_score,
                  t.role_alternative = s.role_alternative,
                  t.role_method = s.role_method,
                  t.category = s.role_category,
                  t.category_score = s.role_score,
                  t.enriched_at = current_timestamp(),
                  t._job_run_id = '{JOB_RUN_ID}'
            """)

        role_counts = {m: sum(1 for v in scored.values() if v["role_method"] == m) for m in ("embedding", "embedding_ambiguous", "llm")}
        waiting_for_embedding = len(need_rows) - len(job_rows)
        metrics.update(rows_in=len(need_rows), rows_out=len(scored), rows_rejected=role_llm_failures + job_embedding_failures)
        metrics["details"].update(
            role_embeddings_new=new_role_embeddings, role_embedding_failures=role_embedding_failures,
            job_embeddings_new=new_job_embeddings, job_embedding_failures=job_embedding_failures,
            waiting_for_embedding=waiting_for_embedding, role_methods=role_counts, role_similarity_rows=similarity_rows,
            role_llm_calls=role_llm_calls, role_llm_cache_hits=role_llm_cache_hits, role_llm_failures=role_llm_failures,
            role_llm_given_up=role_llm_given_up,
        )
        print(f"roles scored={len(scored)} {role_counts} | job embeddings new={new_job_embeddings} failed={job_embedding_failures} "
              f"waiting={waiting_for_embedding} | role LLM calls={role_llm_calls} cache hits={role_llm_cache_hits} "
              f"failed={role_llm_failures} given up={role_llm_given_up}")

## Scope - final eligibility
One pass over every row after role matching (before the skills LLM, so no calls are spent on excluded jobs). The stage 1 reasons (`not_india`, `non_english:*`, title rule) are kept; a job with a role is in or out of scope by that role, so a scope change in `01_setup` also excludes or re-admits jobs enriched earlier. Only rows whose result changes are updated.

In [ ]:
with task_run("enrich_scope") as metrics:
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW scope_updates AS
        WITH base AS (
          SELECT
            e.job_key, e.llm_status, e.skills, length(s.description) AS description_length,
            filter(split(coalesce(e.gold_exclusion_reason, ''), ','), x -> x NOT IN ('', '{SCOPE_REASON}')) AS other_reasons,
            CASE WHEN e.role_title IS NULL
                 THEN array_contains(split(coalesce(e.gold_exclusion_reason, ''), ','), '{SCOPE_REASON}')  -- title rule (stage 1)
                 ELSE r.role_title IS NULL END AS out_of_scope
          FROM {ENRICHED_TABLE} e
          LEFT JOIN in_scope_roles r ON r.role_title = e.role_title
          LEFT JOIN {SILVER_TABLE} s ON s.job_key = e.job_key AND s.description_hash = e.description_hash
        ),
        decided AS (
          SELECT *, CASE WHEN out_of_scope THEN concat(other_reasons, array('{SCOPE_REASON}')) ELSE other_reasons END AS reasons
          FROM base
        )
        SELECT
          job_key,
          size(reasons) = 0 AS gold_eligible,
          nullif(array_join(reasons, ','), '') AS gold_exclusion_reason,
          CASE WHEN size(reasons) > 0 AND llm_status IN ('pending', 'failed') THEN 'skipped'
               WHEN size(reasons) = 0 AND llm_status = 'skipped'  -- re-admitted
                 THEN CASE WHEN size(coalesce(skills, CAST(array() AS ARRAY<STRING>))) < {MIN_SKILLS} AND description_length >= 300 THEN 'pending' ELSE 'not_needed' END
               ELSE llm_status END AS llm_status
        FROM decided
    """)
    result = spark.sql(f"""
        MERGE INTO {ENRICHED_TABLE} t
        USING scope_updates s
        ON t.job_key = s.job_key
        WHEN MATCHED AND NOT (t.gold_eligible <=> s.gold_eligible AND t.gold_exclusion_reason <=> s.gold_exclusion_reason
                              AND t.llm_status <=> s.llm_status)
          THEN UPDATE SET t.gold_eligible = s.gold_eligible, t.gold_exclusion_reason = s.gold_exclusion_reason, t.llm_status = s.llm_status
    """).first().asDict()
    changed = int(result.get("num_updated_rows", 0))
    metrics.update(rows_in=None, rows_out=changed, rows_rejected=0)
    metrics["details"].update(changed=changed, in_scope_roles=len(IN_SCOPE_ROLES))
    print(f"scope: {changed} rows changed eligibility / skills LLM status")

## Stage 3 - skills LLM for eligible jobs with an in-scope role and few skills

In [ ]:
LLM_PROMPT = (
    "You extract structured data from a job posting. Return JSON only.\n"
    "- skills: up to 20 tools, technologies, programming languages, platforms, certifications or domain methods "
    "that are written in the job title or description. Copy each name as it appears, 1 to 4 words. "
    "Never add a skill that is not in the text. "
    "Exclude soft skills (communication, teamwork, leadership), generic words (processes, policies, documentation), "
    "benefits, company names and locations. Return an empty list if none are written.\n"
    "- category: the single best job family from the allowed list.\n"
    "- experience_min_years and experience_max_years: required years of experience as integers, -1 when not stated.\n"
)
# Unchanged from prompt p2 so cached responses stay valid; the category answer is no longer used (roles decide it)
RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "job_enrichment",
        "schema": {
            "type": "object",
            "properties": {
                "skills": {"type": "array", "items": {"type": "string"}},
                "category": {"type": "string", "enum": CATEGORY_NAMES},
                "experience_min_years": {"type": "integer"},
                "experience_max_years": {"type": "integer"},
            },
            "required": ["skills", "category", "experience_min_years", "experience_max_years"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})
assert "'" not in RESPONSE_FORMAT, "response format must not contain single quotes"
LLM_SCHEMA = "struct<skills:array<string>, category:string, experience_min_years:int, experience_max_years:int>"

# LLM skills that are too vague to be useful
GENERIC_SKILLS = [
    "communication", "communication skills", "written communication", "verbal communication", "teamwork", "leadership",
    "problem solving", "problem-solving", "analytical skills", "attention to detail", "documentation", "processes",
    "procedures", "policies", "analytics", "implementation", "reporting", "management",
    "stakeholder management", "time management", "collaboration", "ai", "artificial intelligence",
    "mobile", "cloud", "email", "word", "outlook", "ms office", "microsoft office", "ms office suite", "scripting",
]

llm_calls = llm_failures = llm_applied = llm_cache_hits = 0

with task_run("enrich_llm") as metrics:
    # Pending: eligible (so the role is in scope, see above) with a role, skills LLM due and fewer than MAX_ATTEMPTS calls
    pending_keys = [] if not LLM_ENABLED else (
        spark.table(ENRICHED_TABLE)
        .where(F.col("gold_eligible") & F.col("role_title").isNotNull() & F.col("llm_status").isin("pending", "failed")
               & (F.coalesce("llm_attempts", F.lit(0)) < MAX_ATTEMPTS))
        .join(spark.table(SILVER_TABLE).alias("s").where(RECENT_JOB).select("job_key", "description_hash"),
              ["job_key", "description_hash"])
        .select("job_key", "description_hash", F.when(is_wanted("category"), 0).otherwise(1).alias("priority"))
        .collect()
    )
    if not LLM_ENABLED:
        metrics["message"] = "llm disabled"
    elif not pending_keys:
        metrics.update(rows_in=0, rows_out=0, rows_rejected=0, message="nothing pending")
    else:
        pending = spark.createDataFrame(pending_keys, "job_key STRING, description_hash STRING, priority INT")
        this_prompt = (F.col("endpoint") == LLM_ENDPOINT) & (F.col("prompt_version") == PROMPT_VERSION)

        # 1. Call the LLM only for pending rows without a cached successful response for this endpoint + prompt
        cached_ok = (
            spark.table(LLM_LOG_TABLE).where(this_prompt & F.col("error").isNull() & F.col("response").isNotNull())
            .select("job_key", "description_hash").distinct()
        )
        llm_input = (
            pending.join(cached_ok, ["job_key", "description_hash"], "left_anti")
            .join(
                spark.table(SILVER_TABLE).select(
                    "job_key", "description_hash", "title", "last_seen_at",
                    F.substring(F.coalesce("description", F.lit("")), 1, LLM_MAX_CHARS).alias("description"),
                ),
                ["job_key", "description_hash"],
            )
            .orderBy("priority", F.col("last_seen_at").desc())
            .limit(LLM_MAX_ROWS)
        )
        requests = llm_input.withColumn(
            "request",
            F.concat(F.lit(LLM_PROMPT), F.lit("\nAllowed categories: " + ", ".join(CATEGORY_NAMES)),
                     F.lit("\n\nJob title: "), F.col("title"), F.lit("\n\nJob description:\n"), F.col("description")),
        )
        responses = requests.select(
            "job_key", "description_hash",
            F.expr(f"ai_query('{LLM_ENDPOINT}', request, responseFormat => '{RESPONSE_FORMAT}', failOnError => false)").alias("r"),
        )

        # Persist responses first so the (billable) calls are executed exactly once
        append_llm_log(responses, LLM_ENDPOINT, PROMPT_VERSION)
        call_counts = (
            spark.table(LLM_LOG_TABLE).where((F.col("_job_run_id") == JOB_RUN_ID) & this_prompt)
            .agg(F.count(F.lit(1)).alias("calls"),
                 F.count(F.when(F.col("error").isNotNull() | F.col("response").isNull(), 1)).alias("failures"))
            .first()
        )
        llm_calls, llm_failures = int(call_counts["calls"]), int(call_counts["failures"])

        # 2. Apply the best response per pending row: newest successful one (cached or new), else the newest failure
        best_first = Window.partitionBy("job_key", "description_hash").orderBy(
            (F.col("error").isNull() & F.col("response").isNotNull()).desc(), F.col("called_at").desc()
        )
        calls = (
            spark.table(LLM_LOG_TABLE).where(this_prompt)
            .join(pending, ["job_key", "description_hash"])
            .withColumn("_rank", F.row_number().over(best_first)).where("_rank = 1").drop("_rank")
            .withColumn("parsed", F.from_json("response", LLM_SCHEMA))
            .withColumn("ok", F.col("error").isNull() & F.col("parsed").isNotNull())
            .withColumn("this_run", F.col("_job_run_id") == JOB_RUN_ID)
        )
        llm_cache_hits = calls.where(F.col("ok") & ~F.col("this_run")).count()

        alias_map = F.broadcast(
            spark.table(SKILLS_TABLE)
            .select("skill", "skill_group", F.explode("aliases").alias("alias"))
            .select(F.lower("alias").alias("alias"), F.col("skill").alias("canonical_skill"), "skill_group")
            .dropDuplicates(["alias"])
        )
        implied_map = F.broadcast(spark.createDataFrame(
            [(skill, parent, SKILL_GROUP.get(parent)) for skill, parents in SKILL_IMPLIES.items() for parent in parents],
            "canonical_skill STRING, skill STRING, skill_group STRING",
        ))
        source_text = spark.table(SILVER_TABLE).select(
            "job_key", "description_hash", F.lower(F.concat_ws(" ", "title", "description")).alias("source_text")
        )
        # Skills not in the taxonomy must look technical: contain a capital, digit or symbol, and not be one plain
        # capitalised word ("Drive", "Excellence" from company values boilerplate)
        looks_technical = F.col("raw_skill").rlike(r"[A-Z0-9+#./&]") & ~F.col("raw_skill").rlike(r"^[A-Z][a-z]+$")
        grounded = (
            calls.where("ok")
            .select("job_key", "description_hash", F.posexplode("parsed.skills").alias("pos", "raw_skill"))
            .withColumn("raw_skill", F.trim(F.regexp_replace("raw_skill", r"\s+", " ")))
            .where(F.length("raw_skill").between(2, 30) & (F.size(F.split("raw_skill", " ")) <= 4))
            .where(~F.lower("raw_skill").isin(GENERIC_SKILLS))
            .join(source_text, ["job_key", "description_hash"])
            .where(F.instr("source_text", F.lower("raw_skill")) > 0)  # grounding: must appear in the job text
            .join(alias_map, F.lower("raw_skill") == F.col("alias"), "left")
            .where(F.col("canonical_skill").isNotNull() | looks_technical)
        )
        llm_skills = (
            grounded.select("job_key", "description_hash", "pos", F.coalesce("canonical_skill", "raw_skill").alias("skill"),
                            F.coalesce("skill_group", F.lit("Other")).alias("skill_group"))
            .unionByName(  # implied parent skills, after the skills the LLM found
                grounded.select("job_key", "description_hash", "canonical_skill")
                .join(implied_map, "canonical_skill")
                .select("job_key", "description_hash", F.lit(1000).alias("pos"), "skill", "skill_group")
            )
            .groupBy("job_key", "description_hash")
            .agg(
                F.array_distinct(F.transform(F.array_sort(F.collect_list(F.struct("pos", "skill"))), lambda s: s["skill"])).alias("llm_skills"),
                F.array_distinct(F.collect_list("skill_group")).alias("llm_groups"),
            )
        )
        (
            calls.join(llm_skills, ["job_key", "description_hash"], "left")
            .select(
                "job_key", "description_hash", "ok", "this_run", "llm_skills", "llm_groups",
                F.when(F.col("parsed.experience_min_years").between(0, 40), F.col("parsed.experience_min_years")).alias("llm_min"),
                F.when(F.col("parsed.experience_max_years").between(0, 40), F.col("parsed.experience_max_years")).alias("llm_max"),
            )
            .createOrReplaceTempView("llm_updates")
        )
        years_to_level = """CASE WHEN {v} IS NULL THEN NULL WHEN {v} < 1 THEN 'Entry' WHEN {v} < 3 THEN 'Junior'
                            WHEN {v} < 6 THEN 'Mid' WHEN {v} < 10 THEN 'Senior' ELSE 'Lead/Manager' END"""
        attempts = "coalesce(t.llm_attempts, 0) + CASE WHEN s.this_run THEN 1 ELSE 0 END"
        merge_result = spark.sql(f"""
            MERGE INTO {ENRICHED_TABLE} t
            USING llm_updates s
            ON t.job_key = s.job_key AND t.description_hash = s.description_hash
            WHEN MATCHED AND NOT s.ok THEN UPDATE SET t.llm_status = 'failed', t.llm_attempts = {attempts}, t._job_run_id = '{JOB_RUN_ID}'
            WHEN MATCHED THEN UPDATE SET
              t.skills = slice(array_distinct(concat(coalesce(t.skills, CAST(array() AS ARRAY<STRING>)),
                                                     coalesce(s.llm_skills, CAST(array() AS ARRAY<STRING>)))), 1, 30),
              t.skill_groups = array_distinct(concat(coalesce(t.skill_groups, CAST(array() AS ARRAY<STRING>)),
                                                     coalesce(s.llm_groups, CAST(array() AS ARRAY<STRING>)))),
              t.experience_min_years = coalesce(t.experience_min_years, s.llm_min),
              t.experience_max_years = coalesce(t.experience_max_years, s.llm_max),
              t.experience_level = coalesce(t.experience_level, {years_to_level.format(v='coalesce(t.experience_min_years, s.llm_min)')}),
              t.llm_status = 'done',
              t.llm_attempts = {attempts},
              t.enriched_at = current_timestamp(),
              t._job_run_id = '{JOB_RUN_ID}'
        """).first().asDict()
        llm_applied = int(merge_result.get("num_updated_rows", 0))

        metrics.update(rows_in=llm_calls, rows_out=llm_applied, rows_rejected=llm_failures)
        metrics["details"].update(endpoint=LLM_ENDPOINT, prompt_version=PROMPT_VERSION, max_rows=LLM_MAX_ROWS,
                                  pending=len(pending_keys), cache_hits=llm_cache_hits)

print(f"skills LLM new calls={llm_calls} failures={llm_failures} cache hits={llm_cache_hits} applied={llm_applied}")

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect

In [ ]:
enriched = spark.table(ENRICHED_TABLE)
display(
    enriched.where("gold_eligible")
    .groupBy("role_method", "llm_status").count()
    .orderBy("role_method", "llm_status")
)

In [ ]:
display(
    enriched.where("gold_eligible")
    .groupBy("category", "role_title")
    .agg(F.count("*").alias("jobs"), F.round(F.avg("role_score"), 3).alias("avg_score"),
         F.round(F.avg(F.size("skills")), 1).alias("avg_skills"))
    .orderBy("category", F.col("jobs").desc())
)

In [ ]:
display(
    enriched.join(spark.table(SILVER_TABLE).select("job_key", "source", "company_name", "title"), "job_key")
    .where("gold_eligible AND role_title IS NOT NULL")
    .select("source", "company_name", "title", "role_title", "role_score", "role_alternative", "role_method", "category",
            "experience_level", "experience_min_years", F.size("skills").alias("n_skills"), "skills")
    .orderBy(F.rand(seed=11))
    .limit(30)
)

In [ ]:
# Closest related roles per role (ops.role_similarity, used by the API for related-role fit)
display(
    spark.table(ROLE_SIMILARITY_TABLE).where("role_a <> role_b")
    .withColumn("_rank", F.row_number().over(Window.partitionBy("role_a").orderBy(F.col("sim").desc())))
    .where("_rank <= 3").drop("_rank")
    .orderBy("role_a", F.col("sim").desc())
)

In [ ]:
display(enriched.groupBy("gold_exclusion_reason").count().orderBy(F.col("count").desc()))

In [ ]:
display(spark.table(table("ops", "pipeline_runs")).where("task LIKE 'enrich%'").orderBy(F.col("started_at").desc()).limit(6))